In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)

## Loading the Dataset

We load the Sign Language MNIST dataset using `pandas.read_csv()`.

- `train_df` → Training dataset
- `valid_df` → Validation dataset

Each dataset contains:
- 1 **label column** (gesture class index)
- 784 **pixel columns** (28 × 28 grayscale image flattened)

The validation set is used to measure generalization performance.

In [ ]:
# ----------- Reading the Data -----------

train_df = pd.read_csv("/kaggle/input/sign-language-mnist/sign_mnist_train/sign_mnist_train.csv")
# Load the training dataset from Kaggle input directory
# The file contains:
# - 1 column named "label"
# - 784 columns representing pixel values (28x28 images flattened)

valid_df = pd.read_csv("/kaggle/input/sign-language-mnist/sign_mnist_test/sign_mnist_test.csv")
# Load the validation dataset
# Used to evaluate model performance during training

## Exploring the Dataset

We use `train_df.head()` to inspect the first 5 rows of the dataset.

This allows us to verify:

- The presence of the **label column**
- The total number of feature columns (784 pixels)
- The structure and formatting of the data

Each row represents:
- 1 image (28 × 28 pixels flattened into 784 columns)
- 1 corresponding label (gesture class index)

In [ ]:
# ----------- Exploring the Data -----------

train_df.head()
# Displays the first 5 rows of the training dataset
# Helps us understand:
# - Column structure
# - Label column position
# - Pixel value format

## Extracting Labels and Features

We separate the dataset into:

- **y_train / y_valid** → Target labels (gesture classes)
- **train_df / valid_df** → Feature matrices (pixel values only)

After deleting the `label` column, the dataframes contain only
the 784 pixel intensity values representing the 28 × 28 images.

This separation is required before training a neural network.

In [ ]:
y_train_int = train_df["label"].to_numpy()
y_valid_int = valid_df["label"].to_numpy()

In [ ]:
print(train_df.columns)

In [ ]:
#Extracting the Labels
y_train = train_df['label']
y_valid = valid_df['label']
del train_df['label']
del valid_df['label']

## Extracting Image Data

We convert the feature DataFrames into NumPy arrays using `.values`.

After removing the `label` column, the remaining 784 columns
represent the pixel values of 28 × 28 grayscale images.

Each row in:
- `x_train` → one training image
- `x_valid` → one validation image

The shape should be:

- `(num_samples, 784)`

In [ ]:
# ----------- Extracting the Images -----------

x_train = train_df.values
# Convert the training DataFrame (pixels only) into a NumPy array
# Shape should be (num_samples, 784)

x_valid = valid_df.values
# Convert the validation DataFrame into a NumPy array
# Each row represents one flattened 28x28 image

In [ ]:
x_train.shape, y_train.shape
# x_train.shape → shows dimensions of input feature matrix
# y_train.shape → shows dimensions of label vector

In [ ]:
x_valid.shape, y_valid.shape
# x_valid.shape → shows the shape of validation feature matrix
# y_valid.shape → shows the shape of validation label vector

In [ ]:
# ----------- Visualizing the Data -----------

import matplotlib.pyplot as plt

plt.figure(figsize=(40,40))
# Create a very large figure (40x40 inches)
# This is large because we are plotting 20 images in one row

num_images = 20
# Number of images to display

for i in range(num_images):
    row = x_train[i]
    # Get the i-th training sample (flattened 784 pixels)

    label = y_train[i]
    # Get the corresponding label

    image = row.reshape(28,28)
    # Reshape 784 values back into 28x28 image

    plt.subplot(1, num_images, i+1)
    # Create subplot: 1 row, 20 columns, position i+1

    plt.title(label, fontdict={'fontsize': 30})
    # Set image title as label (gesture class)
    # Increase font size for readability

    plt.axis('off')
    # Hide axis ticks and borders

    plt.imshow(image, cmap='gray')
    # Display image in grayscale

In [ ]:
# ----------- Normalize the Image Data -----------

x_train = x_train.astype("float32") / 255.0
# Scale pixel values from range [0, 255]
# to range [0, 1]
# This improves neural network training stability

x_valid = x_valid.astype("float32") / 255.0
# Apply the same normalization to validation data
# Important: training and validation must be scaled the same way

In [ ]:
# ----------- Categorize the Labels -----------

import tensorflow.keras as keras
# Import Keras from TensorFlow


num_classes = int(y_train_int.max()) + 1
# Define number of output classes
# ⚠ This must match the actual number of unique label indices

In [ ]:


y_train = keras.utils.to_categorical(y_train_int, num_classes=num_classes)
y_valid = keras.utils.to_categorical(y_valid_int, num_classes=num_classes)

print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("num_classes:", num_classes)

## Building a Baseline ANN Classifier

We build a simple fully-connected neural network (ANN) for Sign Language MNIST.

Architecture:
- Dense(512, ReLU)
- Dense(512, ReLU)
- Dense(num_classes, Softmax)

Compilation:
- Optimizer: Adam
- Loss: Categorical Crossentropy (requires one-hot encoded labels)
- Metric: Accuracy

Training:
- 5 epochs
- Validation is evaluated after each epoch to monitor generalization.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential()
model.add(Dense(512, activation="relu", input_shape=(784,)))
model.add(Dense(512, activation="relu"))
model.add(Dense(num_classes, activation="softmax"))  # must match y_train.shape[1]

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

history = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=128,
    validation_data=(x_valid, y_valid),
    verbose=1
)

model.summary()

## Training Curves

We visualize:

- Training accuracy vs Validation accuracy
- Training loss vs Validation loss

These curves help us detect:

- Overfitting → training improves but validation worsens
- Underfitting → both curves stay low
- Good fit → both improve and stay close

In [ ]:
import matplotlib.pyplot as plt

# ----------- Plot Accuracy -----------

plt.figure(figsize=(8,5))

plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Model Accuracy")
plt.legend()
plt.show()


# ----------- Plot Loss -----------

plt.figure(figsize=(8,5))

plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Model Loss")
plt.legend()
plt.show()

## Model Conclusion

The baseline Artificial Neural Network (ANN) demonstrates stable and consistent learning behavior.

### Performance Summary

- Training accuracy improved from ~43% to ~96%.
- Validation accuracy increased from ~56% to ~77%.
- Training loss decreased smoothly from ~1.9 to ~0.2.
- Validation loss decreased overall, with only minor fluctuations.

### Interpretation

The model is clearly learning meaningful patterns from the data.  
Validation performance improves alongside training performance, indicating effective generalization.

However, a noticeable gap (~19%) between training and validation accuracy suggests moderate overfitting.  
The model begins to memorize training data in later epochs but still maintains reasonable validation performance.

### Overall Assessment

- ✅ Stable training behavior  
- ✅ No instability or divergence  
- ✅ Reasonable generalization  
- ⚠ Mild overfitting present  

### Recommendation

While the ANN performs adequately, further improvements could be achieved by:

- Adding regularization (Dropout, L2)
- Using EarlyStopping
- Transitioning to a Convolutional Neural Network (CNN), which is better suited for image data

A CNN architecture is expected to significantly improve validation accuracy due to its ability to capture spatial patterns in images.